In [ ]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
from google.colab import userdata            # Colab's secret store

openai_api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_api_key   # IMPORTANT: the Agents SDK reads the key from here

openai_client = OpenAI(api_key = openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:5])

OpenAI client successfully configured.
sk-pr


In [ ]:
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 279.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 293.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 166.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.8 MB/s eta 0:00:00
^C


In [ ]:
from agents import Agent
market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

market_researcher_agent = Agent(name = "Market Researcher",
                                instructions = market_researcher_instructions,
                                model = "gpt-4.1-mini")


In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

In [ ]:
from google.colab import userdata
from agents import set_tracing_export_api_key

set_tracing_export_api_key(userdata.get("OPENAI_API_KEY"))
# View traces at platform.openai.com/traces

In [ ]:
from agents import Runner,SQLiteSession
q1 = "What is the market share of Tesla in the US EV market?"
resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)
print_markdown(f"🤖 Agent:\n{resp1.final_output}")

q2 = "How does that compare to last year?"
resp2 = await Runner.run(starting_agent = market_researcher_agent, input = q2)
print_markdown(f"🤖 Agent:\n{resp2.final_output}")


🤖 Agent:
✅ FACT: Tesla holds approximately 60-70% of the US electric vehicle (EV) market share as of early 2024, making it the dominant EV manufacturer in the country.

🤖 Agent:
❌ UNKNOWN: You have not provided the specific data or context for this year’s figures to compare with last year.

In [ ]:
# Create a session instance (the agent's conversation diary)
session = SQLiteSession("conversation")

# First interaction
user_input1 = "What is the market share of Tesla in the US EV market?"
response1 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input1,
                             session = session)        # <- NEW
print_markdown(f"🤖 Agent:\n{response1.final_output}")

# Second interaction - same session, so the agent sees the history
user_input2 = "How does that compare to last year?"
response2 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input2,
                             session = session)
print_markdown(f"🤖 Agent:\n{response2.final_output}")


🤖 Agent:
✅ FACT: Tesla holds approximately 60-70% of the US electric vehicle (EV) market share as of early 2024, making it the dominant EV manufacturer in the country.

🤖 Agent:
✅ FACT: Tesla's US EV market share has slightly decreased from around 70% in 2022 to approximately 60-65% in 2023 due to increasing competition from other automakers.